In [12]:
# ============================================================
# RETINAGENT — GRADER AGENT
# ============================================================

import os
import torch
import numpy as np

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_BB_PATH = (
    "/kaggle/usr/lib/notebooks/divyanshukj4495/"
    "imageagent/image_blackboard.pt"
)

CLINICAL_BB_PATH = (
    "/kaggle/usr/lib/notebooks/divyanshukj4495/"
    "clinical_note_agent/clinical_blackboard.pt"
)

GRADER_BB_PATH = "/kaggle/working/grader_blackboard.pt"

os.makedirs("/kaggle/working", exist_ok=True)

print("=" * 70)
print("RETINAGENT - GRADER AGENT")
print("=" * 70)
print("Device:", device)


RETINAGENT - GRADER AGENT
Device: cpu


In [13]:
# ------------------------------------------------------------
# 2. Load ImageAgent Blackboard
# ------------------------------------------------------------

if not os.path.exists(IMAGE_BB_PATH):
    raise FileNotFoundError(
        f"ImageAgent blackboard not found:\n{IMAGE_BB_PATH}"
    )

image_blackboard = torch.load(
    IMAGE_BB_PATH,
    map_location="cpu",
    weights_only=False
)

if "output" not in image_blackboard:
    raise KeyError("ImageAgent blackboard does not contain 'output'.")

image_output = image_blackboard["output"]

print("\nImageAgent output loaded.")


# ------------------------------------------------------------
# 3. Validate ImageAgent Output
# ------------------------------------------------------------

required_image_fields = {
    "grade",
    "confidence",
    "image_embedding",
    "gradcam",
    "lesion_boxes",
    "segmentation_mask",
    "tta_grades",
    "tta_confidences",
    "grade_range",
    "review_required",
}

missing_image_fields = required_image_fields - set(image_output.keys())

if missing_image_fields:
    raise KeyError(
        f"Missing ImageAgent fields: {sorted(missing_image_fields)}"
    )

print("ImageAgent fields validated.")



ImageAgent output loaded.
ImageAgent fields validated.


In [14]:
# ------------------------------------------------------------
# 4. Load Clinical Note Agent Blackboard
# ------------------------------------------------------------

if not os.path.exists(CLINICAL_BB_PATH):
    raise FileNotFoundError(
        f"Clinical Note Agent blackboard not found:\n{CLINICAL_BB_PATH}"
    )

clinical_blackboard = torch.load(
    CLINICAL_BB_PATH,
    map_location="cpu",
    weights_only=False
)

if "output" not in clinical_blackboard:
    raise KeyError(
        "Clinical Note Agent blackboard does not contain 'output'."
    )

clinical_output = clinical_blackboard["output"]

print("Clinical Note Agent output loaded.")


# ------------------------------------------------------------
# 5. Validate Clinical Note Agent Output
# ------------------------------------------------------------

required_clinical_fields = {
    "image_id",
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "diabetes_duration",
    "prior_vitrectomy",
    "symptoms",
    "image_quality",
    "missing_fields",
}

missing_clinical_fields = (
    required_clinical_fields - set(clinical_output.keys())
)

if missing_clinical_fields:
    raise KeyError(
        "Missing Clinical Note Agent fields: "
        f"{sorted(missing_clinical_fields)}"
    )

print("Clinical Note Agent fields validated.")


Clinical Note Agent output loaded.
Clinical Note Agent fields validated.


In [15]:
# ------------------------------------------------------------
# 6. Extract Image Evidence
# ------------------------------------------------------------

image_grade = int(image_output["grade"])
image_confidence = float(image_output["confidence"])

image_embedding = image_output["image_embedding"]

if not isinstance(image_embedding, torch.Tensor):
    image_embedding = torch.tensor(
        image_embedding,
        dtype=torch.float32
    )

image_embedding = image_embedding.detach().float().cpu().flatten()

if image_embedding.numel() != 768:
    raise ValueError(
        f"Expected 768-D ImageAgent embedding, "
        f"got {image_embedding.numel()} dimensions."
    )

tta_grades = [
    int(x)
    for x in image_output["tta_grades"]
]

tta_confidences = [
    float(x)
    for x in image_output["tta_confidences"]
]

print("\nImage embedding:", image_embedding.shape)



Image embedding: torch.Size([768])


In [16]:
# ------------------------------------------------------------
# 7. Encode Clinical Context
# ------------------------------------------------------------
#
# The Clinical Note Agent is independent of the ImageAgent.
# Its fields are preserved as structured clinical evidence.
#
# We do NOT create artificial image-note training pairs.
# We also do NOT use a randomly initialized fusion network
# to generate a fabricated DR prediction.
# ------------------------------------------------------------

def clinical_context_vector(note):
    """
    Compact representation of the clinical-note state.

    This is used as structured context in the Grader output,
    not as a trained DR classifier.
    """

    def present(value):
        return 0.0 if value is None else 1.0

    diabetes_duration = note.get("diabetes_duration")

    try:
        diabetes_duration = float(diabetes_duration)
    except (TypeError, ValueError):
        diabetes_duration = 0.0

    # Normalize duration approximately to [0,1].
    diabetes_duration = min(max(diabetes_duration / 30.0, 0.0), 1.0)

    symptoms = note.get("symptoms", [])

    if symptoms is None:
        symptoms = []

    if not isinstance(symptoms, (list, tuple)):
        symptoms = [symptoms]

    vector = torch.tensor(
        [
            present(note.get("visual_acuity")),
            present(note.get("lens_status")),
            present(note.get("prior_laser")),
            present(note.get("prior_anti_vegf")),
            present(note.get("hba1c")),
            diabetes_duration,
            present(note.get("prior_vitrectomy")),
            1.0 if len(symptoms) > 0 else 0.0,
            present(note.get("image_quality")),
            float(len(note.get("missing_fields", []))) / 7.0,
            1.0 if note.get("image_id") is not None else 0.0,
        ],
        dtype=torch.float32,
    )

    return vector


clinical_vector = clinical_context_vector(clinical_output)

print("Clinical vector:", clinical_vector.shape)


Clinical vector: torch.Size([11])


In [17]:
# ------------------------------------------------------------
# 8. TTA Consistency Gate
# ------------------------------------------------------------
#
# Paper:
# If max(TTA grades) - min(TTA grades) > 1,
# flag the case for clinician review.
# ------------------------------------------------------------

tta_spread = max(tta_grades) - min(tta_grades)

tta_review_required = tta_spread > 1


# ------------------------------------------------------------
# 9. Final Grader Decision
# ------------------------------------------------------------
#
# There is currently no paired multimodal development set
# available in this setup for training W_f.
#
# Therefore:
#
#   Final grade      = ImageAgent grade
#   Final confidence = ImageAgent confidence
#
# Clinical information is retained as contextual evidence.
#
# This avoids producing a meaningless prediction from an
# untrained/random fusion head.
# ------------------------------------------------------------

fused_grade = image_grade
fused_confidence = image_confidence

fusion_status = "IMAGE_GROUNDED_NO_TRAINED_FUSION"
fusion_method = "image_grade_with_clinical_context"


In [19]:
# ------------------------------------------------------------
# 10. Clinical Context Summary
# ------------------------------------------------------------

clinical_available_fields = {}

for key, value in clinical_output.items():

    if key in {"image_id", "missing_fields"}:
        continue

    if value is None:
        continue

    if isinstance(value, (list, tuple)) and len(value) == 0:
        continue

    clinical_available_fields[key] = value


clinical_missing_fields = list(
    clinical_output.get("missing_fields", [])
)


# ------------------------------------------------------------
# 11. Review Decision
# ------------------------------------------------------------

review_required = bool(tta_review_required)

review_reasons = []

if tta_review_required:
    review_reasons.append(
        "TTA grade spread exceeds 1 grade."
    )

if image_confidence < 0.30:
    review_reasons.append(
        "ImageAgent confidence is below 0.30."
    )

if len(clinical_missing_fields) > 0:
    review_reasons.append(
        "Clinical note contains missing structured fields."
    )


# ------------------------------------------------------------
# 12. Grade Labels
# ------------------------------------------------------------

GRADE_LABELS = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR",
}

if fused_grade not in GRADE_LABELS:
    raise ValueError(
        f"Invalid DR grade: {fused_grade}"
    )

grade_label = GRADE_LABELS[fused_grade]


In [20]:
# ------------------------------------------------------------
# 13. Construct Grader Output
# ------------------------------------------------------------

grader_output = {

    "agent": "GraderAgent",

    # Final grading
    "grade": fused_grade,
    "grade_label": grade_label,
    "confidence": fused_confidence,

    # Image evidence
    "image_agent_grade": image_grade,
    "image_agent_confidence": image_confidence,
    "image_embedding": image_embedding,

    # Clinical evidence
    "clinical_context_vector": clinical_vector,
    "clinical_context": clinical_available_fields,
    "clinical_missing_fields": clinical_missing_fields,

    # Explainability evidence passed downstream
    "gradcam": image_output["gradcam"],
    "lesion_boxes": image_output["lesion_boxes"],
    "segmentation_mask": image_output["segmentation_mask"],

    # TTA consistency
    "tta_grades": tta_grades,
    "tta_confidences": tta_confidences,
    "tta_spread": tta_spread,

    # Review
    "review_required": review_required,
    "review_reasons": review_reasons,

    # Fusion metadata
    "fusion_status": fusion_status,
    "fusion_method": fusion_method,

    # Clinical-note identity
    "image_id": clinical_output["image_id"],
}


# ------------------------------------------------------------
# 14. Save Grader Blackboard
# ------------------------------------------------------------

grader_blackboard = {
    "agent": "GraderAgent",
    "output": grader_output,
}

torch.save(
    grader_blackboard,
    GRADER_BB_PATH
)


# ------------------------------------------------------------
# 15. Display Result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GRADER RESULT")
print("=" * 70)

print(
    f"\nImageAgent grade: "
    f"{image_grade} ({GRADE_LABELS[image_grade]})"
)

print(
    f"ImageAgent confidence: "
    f"{image_confidence:.4f}"
)

print(
    f"\nFinal grade: "
    f"{fused_grade} ({grade_label})"
)

print(
    f"Final confidence: "
    f"{fused_confidence:.4f}"
)

print("\nTTA grades:", tta_grades)
print("TTA confidences:", tta_confidences)
print("TTA grade spread:", tta_spread)

print(
    "\nClinical fields available:",
    list(clinical_available_fields.keys())
)

print(
    "Clinical fields missing:",
    clinical_missing_fields
)

print(
    "\nReview required:",
    review_required
)

if review_reasons:
    print("Review reasons:")
    for reason in review_reasons:
        print(" -", reason)

print("\nFusion status:", fusion_status)
print("Fusion method:", fusion_method)

print("\n" + "=" * 70)
print("Grader blackboard saved:")
print(GRADER_BB_PATH)
print("=" * 70)

print("\nRETINAGENT GRADER COMPLETE.")



GRADER RESULT

ImageAgent grade: 0 (No DR)
ImageAgent confidence: 0.3649

Final grade: 0 (No DR)
Final confidence: 0.3649

TTA grades: [0, 0, 0]
TTA confidences: [0.36490851640701294, 0.33550402522087097, 0.41503438353538513]
TTA grade spread: 0

Clinical fields available: ['visual_acuity', 'diabetes_duration', 'image_quality']
Clinical fields missing: ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms']

Review required: False
Review reasons:
 - Clinical note contains missing structured fields.

Fusion status: IMAGE_GROUNDED_NO_TRAINED_FUSION
Fusion method: image_grade_with_clinical_context

Grader blackboard saved:
/kaggle/working/grader_blackboard.pt

RETINAGENT GRADER COMPLETE.
